# protocol

> Claude Code's stream-json wire protocol: NDJSON transport, control routing, and paused MCP tool calls

In [ ]:
#| default_exp protocol

Speak Claude Code's stream-json wire protocol directly, with no Agent SDK: `read_msgs` frames NDJSON from a piped process, `mk_tools` turns annotated callables or schema dicts into the tool list to advertise, `ToolBroker` holds ordinary MCP `tools/call` requests open until the caller supplies their results, `mcp_dispatch` implements the small MCP-shaped JSON-RPC surface the CLI uses, and `ClaudeProto` is the per-process peer. It matches `control_response`s to pending requests, runs each incoming `control_request` as its own cancellable task, answers `control_cancel_request` by cancelling and staying silent, and yields every other message through untouched. `initialize` and `interrupt` are ordinary control requests. Tools are never executed here: the caller owns the tool loop, and the runner in `fastclaude.core` owns the process itself.

In [ ]:
#| export
import asyncio, json, os
from fastcore.utils import *
from fastcore.funccall import get_schema
from fastclaude.session import canon

In [ ]:
from fastcore.test import *
from fastclaude.session import ant_data, sess_dir
from collections import Counter
from contextlib import suppress
import shutil, sys, tempfile

## The stream

Run with `--output-format stream-json --verbose --include-partial-messages` and piped stdio, `claude` becomes a headless NDJSON peer: every line of stdout is one JSON message, and stdin accepts JSON messages the same way. The fixture below is one real captured run: a Bash tool call, made against a scratch project. Like the transcript fixtures in `fastclaude.session`, the capture is checked in as package data, and the builder returns immediately when it exists; delete the file to request a fresh capture against the installed CLI.

In [ ]:
stream_path = ant_data.parent/'stream.jsonl'

In [ ]:
async def mk_stream_fixture(path=None):  # chkstyle: ignore-node
    "Capture one live claude run's raw stream-json stdout as the checked-in stream fixture"
    path = Path(path) if path else stream_path
    if path.exists(): return path
    prompt = 'think hard: first work out 17*23-4 in your head, then use the Bash tool to run: echo flux-41.7 . Then reply with exactly the tool output followed by your arithmetic answer.'
    argv = ['claude','-p','--output-format','stream-json','--verbose','--include-partial-messages','--model','sonnet','--allowedTools','Bash']
    td = tempfile.mkdtemp()
    p = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=td)
    out,_ = await p.communicate(prompt.encode())
    assert not p.returncode, f'claude exited {p.returncode}'
    path.write_bytes(out)
    shutil.rmtree(sess_dir(td), ignore_errors=True)
    return path

In [ ]:
await mk_stream_fixture()

Path('/Users/keremturgutlu/aai-ws/fastclaude/fastclaude/data/stream.jsonl')

A run's stdout mixes several kinds of message. Counting them first shows the shape of a whole conversation before we pick each kind apart:

In [ ]:
evs = dict2obj(stream_path.read_jsonl())
len(evs),Counter(e.type for e in evs)

(26,
 Counter({'stream_event': 16,
          'system': 5,
          'assistant': 2,
          'rate_limit_event': 1,
          'user': 1,
          'result': 1}))

The same content arrives twice. As the model generates, `stream_event` messages wrap the raw Anthropic SSE stream (`message_start`, `content_block_delta`, ...), token by token, for streaming consumers. Then, as each content block completes, a full `assistant` message event carries the finished block; tool results arrive as full `user` message events. The full events match what Claude writes to the session transcript, record for record (same `uuid`s, same `message` content), so a consumer that wants the finished conversation reads only them and ignores the partials:

In [ ]:
tu_ev = first(e for e in evs if e.type=='assistant' and e.message.content[0].type=='tool_use')
tr_ev = first(e for e in evs if e.type=='user')
test_eq(tr_ev.message.content[0].tool_use_id, tu_ev.message.content[0].id)
deltas = [d.event.delta for d in evs if d.type=='stream_event' and d.event.type=='content_block_delta']
test_eq(json.loads(''.join(d.partial_json for d in deltas if d.type=='input_json_delta')), obj2dict(tu_ev.message.content[0].input))
tu_ev.message.content[0].name, tr_ev.message.content[0].content

('Bash', 'flux-41.7')

The rest is ambient noise a reader must tolerate rather than parse: `system` events (`init` with the session's tools and config, `status`, `thinking_tokens`, and `hook_started`/`hook_response` when the user's own hooks fire, which they do even headless), `rate_limit_event`, `command_lifecycle` and `attachment` records on a resume, and whatever new types later CLI versions add. The protocol layer therefore never enumerates event types: it routes the three control messages it owns and passes everything else through untouched.

## NDJSON framing

Pipes deliver chunks, not lines: one read can return half a message, or three. asyncio's `StreamReader.readline` owns the reassembly, bounded by the `limit` passed when the process is spawned (one claude message can carry megabytes of base64 image, so the runner spawns with a generous limit rather than the 64KB default). `read_msgs` turns the byte stream into decoded messages, skipping blank lines and failing loudly on anything that is not JSON: silently dropping a line would hide a desynced stream.

In [ ]:
#| export
async def read_msgs(
    stream, # An asyncio `StreamReader` of NDJSON, e.g. a claude process's stdout
):
    "Decoded messages from `stream`, one per line; blank lines skip, non-JSON raises, a truncated final fragment drops"
    while line := await stream.readline():
        if not (s := line.strip()): continue
        try: yield json.loads(s)
        except json.JSONDecodeError as e:
            if line.endswith(b'\n'): raise ValueError(f'bad NDJSON line: {s[:200]!r}') from e
            return  # no newline: a producer killed mid-write; the fragment is unrecoverable

A scripted producer replays the captured stream in aggressive 37-byte writes, so nearly every message arrives split across reads. Framing reassembles exactly the events the capture holds:

In [ ]:
emit = f"""import sys
data = open({str(stream_path)!r}, 'rb').read()
for i in range(0, len(data), 37):
    sys.stdout.buffer.write(data[i:i+37])
    sys.stdout.buffer.flush()"""
p = await asyncio.create_subprocess_exec(sys.executable, '-c', emit, stdout=asyncio.subprocess.PIPE, limit=2**25)
got = [m async for m in read_msgs(p.stdout)]
await p.wait()
test_eq(got, list(obj2dict(evs)))
test_eq(got[-1]['type'], 'result')

Blank lines skip, a complete final line missing only its newline still arrives, a fragment truncated mid-write is dropped (only a killed producer leaves one), and a non-JSON line raises rather than desyncing:

In [ ]:
def feedr(b):
    "A `StreamReader` pre-fed with `b`, at EOF"
    r = asyncio.StreamReader()
    r.feed_data(b)
    r.feed_eof()
    return r

test_eq([m async for m in read_msgs(feedr(b'{"a": 1}\n\n{"b": 2}'))], [dict(a=1), dict(b=2)])
test_eq([m async for m in read_msgs(feedr(b'{"a": 1}\n{"cut": tru'))], [dict(a=1)])
bad = read_msgs(feedr(b'not json\n'))
with expect_fail(ValueError, contains='bad NDJSON'): await bad.__anext__()

## Tool schemas


A tool is a schema to advertise, nothing more: the caller executes it, so this layer never holds a callable. A schema dict carries `name`, `description`, and `inputSchema`. An ordinary annotated Python function is accepted as sugar: `get_schema` derives the same dict from its signature and docstring, and the function itself is never called. `mk_tools` normalizes a mixed list of both forms:


In [ ]:
#| export
def tool_spec(
    t, # An annotated callable, or a schema dict with `name`, `description`, `inputSchema`
):
    "The schema dict for one tool; a callable's schema is derived from its signature, and the callable is never executed"
    return get_schema(t, pname='inputSchema') if callable(t) else t

def mk_tools(
    tools, # Tools in either `tool_spec` form
):
    "The schema list to advertise for `tools`"
    return [tool_spec(t) for t in listify(tools)]

In [ ]:
async def flux_meter(unit:str='kf') -> str:
    "Read the flux."
    return f'flux: 41.7 {unit}'

py_schema = dict(name='py', description='Run code in the kernel',
    inputSchema=dict(type='object', properties=dict(code=dict(type='string')), required=['code']))
schemas = mk_tools([flux_meter, py_schema])
test_eq([s['name'] for s in schemas], ['flux_meter','py'])
schemas[0]

{'name': 'flux_meter',
 'description': 'Read the flux.\n\nReturns:\n- type: string',
 'inputSchema': {'type': 'object',
  'properties': {'unit': {'description': '',
    'default': 'kf',
    'type': 'string'}}}}

## Paused tool calls

Anthropic tool results and MCP tool returns use nearly the same content blocks, but their image fields differ: Anthropic nests the media type and data under `source`, while MCP puts `mimeType` and `data` directly on the block. `_mcp_content` makes that boundary explicit and leaves already-compatible blocks unchanged:

In [ ]:
#| export
def _mcp_content(block):
    "Translate one Anthropic content block to MCP content"
    if block.get('type')!='image': return block
    src = block['source']
    return dict(type='image', data=src['data'], mimeType=src['media_type'])

In [ ]:
text_block = dict(type='text', text='flux: 41.7 gauss')
image_block = dict(type='image', source=dict(type='base64', media_type='image/png', data='aGVsbG8='))
test_eq(_mcp_content(text_block), text_block)
converted_image = _mcp_content(image_block)
test_eq(converted_image, dict(type='image', mimeType='image/png', data='aGVsbG8='))
converted_image

{'type': 'image', 'data': 'aGVsbG8=', 'mimeType': 'image/png'}

`tool_reply` then has only three jobs: accept the convenient string form, translate each content block through `_mcp_content`, and attach MCP's `isError` flag.

In [ ]:
#| export
def tool_reply(content, is_error=False):
    "An MCP tool return value carrying Anthropic content"
    if isinstance(content, str): content = [dict(type='text', text=content)]
    return dict(content=[_mcp_content(b) for b in content], isError=is_error)

In [ ]:
reply = tool_reply([text_block, image_block], is_error=True)
test_eq(reply['content'], [text_block, converted_image])
test_eq(reply['isError'], True)
reply

{'content': [{'type': 'text', 'text': 'flux: 41.7 gauss'},
  {'type': 'image', 'data': 'aGVsbG8=', 'mimeType': 'image/png'}],
 'isError': True}

The model stream and the MCP request identify a call differently. The stream has Claude's call id; the request has only a name and arguments. `_call_key` canonicalizes the latter pair so dictionary order cannot affect matching.

In [ ]:
#| export
def _call_key(name, arguments): return canon([name, canon(dict(arguments))])

In [ ]:
k1 = _call_key('flux_meter', dict(unit='gauss', scale=1))
k2 = _call_key('flux_meter', dict(scale=1, unit='gauss'))
test_eq(k1, k2)
k1

'["flux_meter","{\\"scale\\":1,\\"unit\\":\\"gauss\\"}"]'

An MCP request can arrive before or after its streamed tool use. `_McpCall` holds the waiting request and its result future; its `slot` is filled only after the two views have been matched.

In [ ]:
#| export
class _McpCall:
    def __init__(self, name, arguments):
        self.name,self.arguments,self.slot = name,arguments,None
        self.future = asyncio.get_running_loop().create_future()

In [ ]:
mcp_call = _McpCall('flux_meter', dict(unit='gauss'))
(mcp_call.name, mcp_call.arguments, mcp_call.slot, mcp_call.future.done())

('flux_meter', {'unit': 'gauss'}, None, False)

Conversely, `_ToolSlot` begins with Claude's id, name, and arguments. It may receive either its matching MCP request or its caller-supplied result first.

In [ ]:
#| export
class _ToolSlot:
    def __init__(self, use):
        self.id,self.name,self.arguments = use.id,use.name,use.arguments
        self.call,self.result = None,None

In [ ]:
streamed_use = AttrDict(id='call_gauss', name='flux_meter', arguments=dict(unit='gauss'))
tool_slot = _ToolSlot(streamed_use)
(tool_slot.id, tool_slot.name, tool_slot.arguments, tool_slot.call, tool_slot.result)

('call_gauss', 'flux_meter', {'unit': 'gauss'}, None, None)

`ToolBroker` owns the two arrival queues and an event used by its process driver. `pending` deliberately excludes requests whose result future is already complete.

In [ ]:
#| export
class ToolBroker:
    "Match a streamed tool batch with the MCP calls collecting its results"
    def __init__(self): self.calls,self.batch,self.changed = [],[],asyncio.Event()

    @property
    def pending(self): return [c for c in self.calls if not c.future.done()]

    @property
    def batch_complete(self): return bool(self.batch) and all(s.call and s.result is not None for s in self.batch)

    async def wait(self):
        "Wait until the pending MCP calls change"
        await self.changed.wait()
        self.changed.clear()

In [ ]:
broker = ToolBroker()
(len(broker.calls), len(broker.batch), len(broker.pending), broker.changed.is_set())

(0, 0, 0, False)

When several identical tools are requested, `_slot` selects the first still-unbound matching slot. That gives identical calls FIFO behavior without assuming that different calls arrive in stream order.

In [ ]:
#| export
@patch
def _slot(self:ToolBroker, call):
    return first(s for s in self.batch
        if s.call is None and _call_key(s.name, s.arguments)==_call_key(call.name, call.arguments))

In [ ]:
broker.batch = [tool_slot]
test_is(broker._slot(mcp_call), tool_slot)
broker._slot(mcp_call).id

'call_gauss'

`_attach` makes the association in both directions. A result may already be waiting in the slot, in which case attachment also completes the MCP future immediately.

In [ ]:
#| export
@patch
def _attach(self:ToolBroker, call):
    slot = self._slot(call)
    if slot is None: raise ValueError(f'unexpected MCP tool call: {call.name} {call.arguments}')
    call.slot = slot
    slot.call = call
    if slot.result is not None: call.future.set_result(slot.result)

In [ ]:
broker._attach(mcp_call)
test_is(mcp_call.slot, tool_slot)
test_is(tool_slot.call, mcp_call)
(tool_slot.id, mcp_call.future.done())

('call_gauss', False)

`call` represents the CLI side of the rendezvous. It registers the request and attaches immediately only while the current streamed batch is unfinished. If the retained batch is complete, the request remains suspended for the next `begin`; cancellation always removes it from the broker.

In [ ]:
#| export
@patch
async def call(self:ToolBroker, name, arguments):
    "Register one MCP call and wait for its result"
    call = _McpCall(name, arguments)
    self.calls.append(call)
    if self.batch and not self.batch_complete: self._attach(call)
    self.changed.set()
    try: return await call.future
    finally:
        self.calls.remove(call)
        self.changed.set()

The other side begins when the model's complete streamed tool-use batch is known. `begin` replaces an absent or completed batch, creates the new slots, and attaches MCP requests that arrived early; it refuses to discard an unfinished earlier batch.

In [ ]:
#| export
@patch
def begin(self:ToolBroker, uses):
    "Install the complete streamed tool-use batch"
    if self.batch and not self.batch_complete: raise RuntimeError('previous tool batch is incomplete')
    self.batch = [_ToolSlot(u) for u in uses]
    for call in self.pending: self._attach(call)
    return self.batch

`reply` addresses a result by Claude's call id. Storing it in the slot first is what makes serialized MCP submission harmless: if that request has not arrived yet, attachment will deliver the waiting result later.

In [ ]:
#| export
@patch
def reply(self:ToolBroker, call_id, content, is_error=False):
    "Store one result and serve it when its MCP call is attached"
    slot = first(s for s in self.batch if s.id==call_id)
    if slot is None: raise ValueError(f'no tool call in this batch: {call_id}')
    if slot.result is not None: raise ValueError(f'tool call already answered: {call_id}')
    slot.result = tool_reply(content, is_error)
    if slot.call is not None: slot.call.future.set_result(slot.result)

Closing a broker has no synthetic success path: every unanswered MCP request is cancelled.

In [ ]:
#| export
@patch
def close(self:ToolBroker):
    "Cancel every pending MCP call"
    for call in self.pending: call.future.cancel()

The rendezvous is easiest to see in arrival order. First an MCP request arrives before the streamed batch, so the request is visible and its task remains suspended.

In [ ]:
broker = ToolBroker()
gauss = asyncio.create_task(broker.call('flux_meter', dict(unit='gauss')))
await broker.wait()
[(c.name, c.arguments, c.future.done()) for c in broker.pending]

[('flux_meter', {'unit': 'gauss'}, False)]

The model stream then reveals the complete two-call batch. `begin` attaches the waiting gauss request, while the tesla slot remains available for the CLI request that has not arrived yet.

In [ ]:
uses = [AttrDict(id=f'call_{unit}', name='flux_meter', arguments=dict(unit=unit)) for unit in ('gauss','tesla')]
slots = broker.begin(uses)
[(s.id, s.call is not None, s.result is not None) for s in slots]

[('call_gauss', True, False), ('call_tesla', False, False)]

The caller answers the whole parallel batch at once. Gauss completes its waiting MCP request immediately; tesla's result stays in its unmatched slot.

In [ ]:
for use in uses: broker.reply(use.id, f"flux: 41.7 {use.arguments['unit']}")
gauss_reply = await gauss
[(s.id, s.call is not None, s.result['content'][0]['text']) for s in slots]

[('call_gauss', True, 'flux: 41.7 gauss'),
 ('call_tesla', False, 'flux: 41.7 tesla')]

When the CLI eventually submits the serialized tesla request, attachment finds the stored result and returns without another model or caller round trip.

In [ ]:
tesla_reply = await broker.call('flux_meter', dict(unit='tesla'))
replies = [gauss_reply['content'][0]['text'], tesla_reply['content'][0]['text']]
test_eq(replies, ['flux: 41.7 gauss','flux: 41.7 tesla'])
replies

['flux: 41.7 gauss', 'flux: 41.7 tesla']

A completed batch may be followed immediately by another tool round. The next MCP request can again arrive before its streamed `tool_use`; it must wait for the next `begin` rather than trying to attach to the completed batch.

In [ ]:
weber = asyncio.create_task(broker.call('flux_meter', dict(unit='weber')))
await asyncio.sleep(0)
if weber.done(): await weber
weber_use = AttrDict(id='call_weber', name='flux_meter', arguments=dict(unit='weber'))
broker.begin([weber_use])
broker.reply(weber_use.id, 'flux: 41.7 weber')
weber_reply = await weber
test_eq(weber_reply['content'][0]['text'], 'flux: 41.7 weber')
weber_reply

{'content': [{'type': 'text', 'text': 'flux: 41.7 weber'}], 'isError': False}

The JSON-RPC envelope is the same three keys every time, so `jrpc` builds one message from its method, id, and params. A None id makes a notification:

In [ ]:
#| export
def jrpc(
    method, # JSON-RPC method name
    id=None, # Request id; None makes a notification
    **params, # The call's `params`, omitted when empty
):
    "One JSON-RPC message, e.g. what the CLI sends our bridge"
    r = dict(jsonrpc='2.0', id=id, method=method)
    if id is None: r.pop('id')
    if params: r['params'] = params
    return r

In [ ]:
test_eq(jrpc('ping', 2), dict(jsonrpc='2.0', id=2, method='ping'))
jrpc('tools/call', 9, name='flux_meter', arguments=dict(unit='gauss'))

{'jsonrpc': '2.0',
 'id': 9,
 'method': 'tools/call',
 'params': {'name': 'flux_meter', 'arguments': {'unit': 'gauss'}}}

## The MCP-shaped bridge

Claude Code talks to the tools we advertise in MCP's JSON-RPC shapes, nested inside its control protocol (the next section). The dispatcher implements exactly the methods the CLI uses—`initialize`, `notifications/*`, `ping`, `tools/list`, and `tools/call`—and nothing else. It is deliberately not a general MCP implementation, and `fastclaude` does not depend on the `mcp` package.

`tools/call` awaits the broker. That wait is the pause: the same CLI process and the same MCP request stay alive while a Responses client executes the function and returns its output.


In [ ]:
#| export
async def mcp_dispatch(
    msg, # One JSON-RPC message from the CLI
    schemas, # Tool schemas to advertise, from `mk_tools`
    broker, # Pending tool calls, supplied later through `ToolBroker.reply`
    server='fastclaude', # Server name reported to the CLI
):
    "The JSON-RPC response for `msg`, or None for a notification"
    m,i = msg.get('method'), msg.get('id')
    def res(r): return dict(jsonrpc='2.0', id=i, result=r)
    if m == 'initialize':
        pv = nested_idx(msg, 'params', 'protocolVersion') or '2024-11-05'
        return res(dict(protocolVersion=pv, capabilities=dict(tools={}), serverInfo=dict(name=server, version='1.0')))
    if m and m.startswith('notifications/'): return None
    if m == 'ping': return res({})
    if m == 'tools/list': return res(dict(tools=schemas))
    if m == 'tools/call':
        nm,args = nested_idx(msg, 'params', 'name'), nested_idx(msg, 'params', 'arguments') or {}
        return res(await broker.call(nm, args))
    return dict(jsonrpc='2.0', id=i, error=dict(code=-32601, message=f'method not found: {m}'))

`disp` binds the schemas and a fresh broker, so the handshake and listing read as direct protocol calls. The handshake echoes the client's protocol version, a notification returns None (the outer control request is still acknowledged in the next section), and the tool listing is exactly the advertised schemas:


In [ ]:
schemas = mk_tools([flux_meter, py_schema])
broker = ToolBroker()
async def disp(method, id=None, **p): return await mcp_dispatch(jrpc(method, id, **p), schemas, broker)
r = await disp('initialize', 1, protocolVersion='2025-06-18')
test_eq(r['result']['protocolVersion'], '2025-06-18')
test_eq(await disp('notifications/initialized'), None)
test_eq((await disp('tools/list', 3))['result']['tools'], schemas)
[m['name'] for m in schemas]


['flux_meter', 'py']

A `tools/call` does not answer merely because it arrived. The pending request becomes visible to the caller; binding gives it Claude's `call_id`, and only `reply` lets the JSON-RPC response complete:


In [ ]:
flux_task = asyncio.create_task(disp('tools/call', 9, name='flux_meter', arguments=dict(unit='gauss')))
await broker.wait()
call = broker.pending[0]
test_eq((call.name, call.arguments), ('flux_meter', dict(unit='gauss')))
broker.begin([AttrDict(id='call_9', name='flux_meter', arguments=dict(unit='gauss'))])
broker.reply('call_9', 'flux: 41.7 gauss')
flux_result = (await flux_task)['result']
test_eq(flux_result['content'][0]['text'], 'flux: 41.7 gauss')
flux_result

{'content': [{'type': 'text', 'text': 'flux: 41.7 gauss'}], 'isError': False}

Cancelling the waiting dispatcher removes the call without fabricating a tool result. An unknown JSON-RPC method remains a normal protocol error:


In [ ]:
cancelled = asyncio.create_task(disp('tools/call', 10, name='flux_meter', arguments={}))
while not broker.pending: await broker.wait()
cancelled.cancel()
with suppress(asyncio.CancelledError): await cancelled
test_eq(broker.pending, [])
test_eq((await disp('resources/list', 4))['error']['code'], -32601)


## Control routing

The three envelope shapes are one-liners, used by the peer below, by test peers, and by anything else that speaks the wire:

In [ ]:
#| export
def ctrl_req(rid, **req):
    "A `control_request` envelope"
    return dict(type='control_request', request_id=rid, request=req)

def ctrl_ok(rid, **resp):
    "A success `control_response` envelope"
    return dict(type='control_response', response=dict(subtype='success', request_id=rid, response=resp))

def ctrl_err(rid, error):
    "An error `control_response` envelope"
    return dict(type='control_response', response=dict(subtype='error', request_id=rid, error=str(error)))

Control traffic rides the same NDJSON stream as everything else, as three message types. A `control_request` carries a `request_id` and a request dict: Claude sends one to reach our bridge (`subtype: mcp_message`, wrapping one JSON-RPC message), while we send one for the handshake (`subtype: initialize`) or a native interrupt. A `control_response` answers one by id. A `control_cancel_request` says Claude abandoned a request it made: the handler is cancelled, its broker call disappears, and no response is written.

`ClaudeProto` is the peer for one process. It matches responses to pending requests, spawns one tracked task per incoming request so a waiting tool never blocks the stream reader, and yields every non-control message untouched. It registers no tool hook: an MCP handler waiting for its result is Claude's ordinary pause mechanism. A JSON-RPC notification returns nothing inner, but the outer control request still gets an acknowledgement, or Claude would wait on it forever.


In [ ]:
#| export
class ClaudeProto:
    "Control-protocol peer for one claude process: request matching, MCP routing, and passthrough events"
    def __init__(self,
        proc, # An asyncio subprocess speaking stream-json on piped stdin/stdout
        tools=None, # Tool schemas to advertise, in either `tool_spec` form
        broker=None, # Broker whose MCP calls wait for caller-supplied results
        server='fastclaude', # SDK MCP server name, matching the `--mcp-config` entry
    ):
        self.proc,self.broker,self.server = proc,broker or ToolBroker(),server
        self.schemas = mk_tools(tools or [])
        self._lock,self._n,self._pending,self._inflight = asyncio.Lock(),0,{},{}

    async def send(self, obj):
        "Write one JSON message to claude's stdin"
        async with self._lock:
            self.proc.stdin.write(json.dumps(obj, ensure_ascii=False).encode()+b'\n')
            await self.proc.stdin.drain()

    async def send_req(self, req, timeout=60):
        "Send a control request, await its response by id, and return the inner `response` dict"
        self._n += 1
        rid = f'req_{self._n}_{os.urandom(4).hex()}'
        fut = asyncio.get_running_loop().create_future()
        self._pending[rid] = fut
        await self.send(ctrl_req(rid, **req))
        try: return await asyncio.wait_for(fut, timeout)
        finally: self._pending.pop(rid, None)

    async def initialize(self, timeout=120):
        "Complete the control-protocol handshake"
        return await self.send_req(dict(subtype='initialize'), timeout)

    async def interrupt(self, timeout=30):
        "Claude's native interrupt: end the current turn, keeping the process alive"
        return await self.send_req(dict(subtype='interrupt'), timeout)

In [ ]:
#| export
@patch
def _resolve(self:ClaudeProto, msg):
    "Complete the pending request a `control_response` answers"
    r = msg.get('response') or {}
    if (fut := self._pending.get(r.get('request_id'))) and not fut.done():
        if r.get('subtype')=='error': fut.set_exception(RuntimeError(r.get('error') or 'control request failed'))
        else: fut.set_result(r.get('response') or {})

@patch
async def _handle(self:ClaudeProto, rid, req):
    "Answer one CLI-originated control request; cancelled handlers answer nothing"
    try:
        if req.get('subtype')!='mcp_message': raise ValueError(f"unsupported control request: {req.get('subtype')}")
        msg = req.get('message') or {}
        mcp_response = await mcp_dispatch(msg, self.schemas, self.broker, self.server)
        await self.send(ctrl_ok(rid, mcp_response=mcp_response or dict(jsonrpc='2.0', result={})))
    except asyncio.CancelledError: raise
    except Exception as e: await self.send(ctrl_err(rid, e))

The read loop ties the pieces together; ending it, from either side, runs `aclose`:

In [ ]:
#| export
@patch
async def events(self:ClaudeProto):
    "Non-control messages from claude, with control traffic routed internally"
    try:
        async for m in read_msgs(self.proc.stdout):
            t = m.get('type')
            if t=='control_response': self._resolve(m)
            elif t=='control_request':
                rid = m.get('request_id')
                task = asyncio.create_task(self._handle(rid, m.get('request') or {}))
                self._inflight[rid] = task
                task.add_done_callback(lambda _,rid=rid: self._inflight.pop(rid, None))
            elif t=='control_cancel_request':
                if task := self._inflight.pop(m.get('request_id'), None): task.cancel()
            else: yield m
    finally: await self.aclose()


In [ ]:
#| export
@patch
async def aclose(self:ClaudeProto):
    "Cancel in-flight handlers, pending requests, and unanswered MCP calls"
    self.broker.close()
    for t in list(self._inflight.values()): t.cancel()
    for f in self._pending.values():
        if not f.done(): f.cancel()

A scripted peer in memory exercises the whole contract without a process or model call. Its stdout is a `StreamReader` fed by hand; its stdin is a sink collecting what `ClaudeProto` writes back. The broker begins empty:


In [ ]:
class _Sink:
    "Collects written messages; a stand-in for a process stdin"
    def __init__(self): self.msgs = []
    def write(self, b): self.msgs.append(json.loads(b))
    async def drain(self): pass

rdr = asyncio.StreamReader()
peer = AttrDict(stdout=rdr, stdin=_Sink())
broker = ToolBroker()
proto = ClaudeProto(peer, tools=[flux_meter], broker=broker)
def feed(o): rdr.feed_data(json.dumps(o).encode()+b'\n')


The events consumer drains in its own task while the scripted CLI sends control traffic into the same stream:

In [ ]:
out = []
async def drain():
    async for m in proto.events(): out.append(m)
t = asyncio.create_task(drain())


A cancelled MCP request should disappear rather than fabricate a tool result. The CLI sends a `control_cancel_request` naming the enclosing control request; the broker observes the waiting call arrive and then its removal:

In [ ]:
feed(ctrl_req('cancel', subtype='mcp_message', server_name='fastclaude',
    message=jrpc('tools/call', 0, name='flux_meter', arguments=dict(unit='cancelled'))))
await broker.wait()
test_eq(len(broker.pending), 1)
feed(dict(type='control_cancel_request', request_id='cancel'))
await broker.wait()
test_eq(broker.pending, [])
peer.stdin.msgs

[]

The first real MCP call then waits without producing a control response. Once the streamed two-call batch arrives, both caller results can be installed even though only the gauss request is attached.

In [ ]:
feed(ctrl_req('m1', subtype='mcp_message', server_name='fastclaude',
    message=jrpc('tools/call', 1, name='flux_meter', arguments=dict(unit='gauss'))))
await broker.wait()
test_eq(peer.stdin.msgs, [])
uses = [AttrDict(id=f'call_{unit}', name='flux_meter', arguments=dict(unit=unit)) for unit in ('gauss','tesla')]
broker.begin(uses)
for use in uses: broker.reply(use.id, f"flux: 41.7 {use.arguments['unit']}")
await broker.wait()

Finally the CLI submits its serialized tesla call. Its waiting result is returned, then the scripted peer sends the terminal event and closes the stream.

In [ ]:
feed(ctrl_req('m2', subtype='mcp_message', server_name='fastclaude',
    message=jrpc('tools/call', 2, name='flux_meter', arguments=dict(unit='tesla'))))
await asyncio.sleep(0.05)
feed(dict(type='result', subtype='success'))
rdr.feed_eof()
await t

Both serialized MCP responses carry their matching batch result, while the cancelled request remained silent:

In [ ]:
resp = {m['response']['request_id']: m['response']['response'] for m in peer.stdin.msgs}
test_eq(resp['m1']['mcp_response']['result']['content'][0]['text'], 'flux: 41.7 gauss')
test_eq(resp['m2']['mcp_response']['result']['content'][0]['text'], 'flux: 41.7 tesla')
test_eq('cancel' in resp, False)
test_eq(broker.pending, [])
test_eq([m['type'] for m in out], ['result'])
resp


{'m1': {'mcp_response': {'jsonrpc': '2.0',
   'id': 1,
   'result': {'content': [{'type': 'text', 'text': 'flux: 41.7 gauss'}],
    'isError': False}}},
 'm2': {'mcp_response': {'jsonrpc': '2.0',
   'id': 2,
   'result': {'content': [{'type': 'text', 'text': 'flux: 41.7 tesla'}],
    'isError': False}}}}

## A live handshake

The live boundary that matters is a parallel batch. Sonnet 5 streams two ToolUses and `message_stop` while the first MCP handler waits, but submits the MCP calls themselves one at a time. Both client results enter the broker together; answering the first releases the second call, whose already-known result returns immediately, and the same process continues to its final answer.

In [ ]:
#| eval: false
td = tempfile.mkdtemp()
server = dict(mcpServers=dict(fastclaude=dict(type='sdk', name='fastclaude')))
argv = ['claude','--output-format','stream-json','--input-format','stream-json','--verbose','--include-partial-messages',
    '--model','claude-sonnet-5','--mcp-config',json.dumps(server),'--strict-mcp-config','--tools','',
    '--allowedTools','mcp__fastclaude__flux_meter']
env = dict(os.environ, ANTHROPIC_API_KEY='')
env.pop('CLAUDECODE', None)
argv

['claude',
 '--output-format',
 'stream-json',
 '--input-format',
 'stream-json',
 '--verbose',
 '--include-partial-messages',
 '--model',
 'claude-sonnet-5',
 '--mcp-config',
 '{"mcpServers": {"fastclaude": {"type": "sdk", "name": "fastclaude"}}}',
 '--strict-mcp-config',
 '--tools',
 '',
 '--allowedTools',
 'mcp__fastclaude__flux_meter']

The live probe starts the CLI directly so the notebook exposes the exact protocol claim rather than relying on `ClaudeRun`. The process uses the private SDK-shaped MCP server and no built-in tools.

In [ ]:
#| eval: false
lp = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE,
    stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=td, env=env)
lbroker = ToolBroker()
lproto = ClaudeProto(lp, tools=[flux_meter], broker=lbroker)
levs,stopped = [],asyncio.Event()
(lp.pid, lproto.schemas)

(17180,
 [{'name': 'flux_meter',
   'description': 'Read the flux.\n\nReturns:\n- type: string',
   'inputSchema': {'type': 'object',
    'properties': {'unit': {'description': '',
      'default': 'kf',
      'type': 'string'}}}}])

The event consumer records everything, marks the model's `message_stop` boundary independently of the terminal result, and keeps draining until the process actually finishes.

In [ ]:
async def _record_live_events(proto, events, stopped):
    async for m in proto.events():
        events.append(m)
        if m.get('type')=='stream_event' and nested_idx(m, 'event', 'type')=='message_stop': stopped.set()
        if m.get('type')=='result': return

With the consumer active, the ordinary control handshake precedes the one user message. The prompt explicitly asks for a parallel batch so the recorded boundary is unambiguous.

In [ ]:
#| eval: false
lt = asyncio.create_task(_record_live_events(lproto, levs, stopped))
await lproto.initialize()
prompt = 'Call flux_meter twice in the same response: once with unit="gauss" and once with unit="tesla". Issue both calls together before either result. Then reply with both outputs.'
await lproto.send(dict(type='user', message=dict(role='user', content=prompt)))

The model reaches `message_stop` with both tool uses streamed, while only the first serialized MCP request is pending. This is the ordering that makes the broker necessary.

In [ ]:
#| eval: false
await asyncio.wait_for(stopped.wait(), 60)
assistant_blocks = [b for m in levs if m.get('type')=='assistant' for b in nested_idx(m, 'message', 'content') or []]
luses = [AttrDict(id=b['id'], name=b['name'].removeprefix('mcp__fastclaude__'), arguments=b['input'])
    for b in assistant_blocks if b.get('type')=='tool_use']
test_eq(sorted(u.arguments['unit'] for u in luses), ['gauss','tesla'])
test_eq(len(lbroker.pending), 1)
lbroker.begin(luses)
[(u.id, u.name, u.arguments) for u in luses]

[('toolu_01Q1772BE6eDqUNqAS49HVB3', 'flux_meter', {'unit': 'gauss'}),
 ('toolu_011Fh3d3yKViv1EhnjPjwrxh', 'flux_meter', {'unit': 'tesla'})]

Supplying both results releases the serialized MCP calls and lets the same process continue to its terminal answer. The result must contain both values; only then is the live probe complete.

In [ ]:
#| eval: false
for use in luses: lbroker.reply(use.id, f"flux: 41.7 {use.arguments['unit']}")
await asyncio.wait_for(lt, 60)
lresult = first(m for m in levs if m.get('type')=='result')
test('41.7 gauss', lresult['result'], in_)
test('41.7 tesla', lresult['result'], in_)
lresult['result']

'Results:\n- gauss: flux: 41.7 gauss\n- tesla: flux: 41.7 tesla'

The probe then closes its stdin, reaps the process, and removes both its transient transcript location and working directory.

In [ ]:
#| eval: false
lp.stdin.close()
await lp.wait()
shutil.rmtree(sess_dir(td), ignore_errors=True)
shutil.rmtree(td, ignore_errors=True)

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()